In [2]:
import ast
import sqlite3
import pandas as pd

CSV = "IT Support Ticket Data.csv"       

df = pd.read_csv(CSV)
df = df[["Body", "Department", "Priority", "Tags"]].dropna(subset=["Department"])
df = df.reset_index(drop=True)
df["ticket_id"] = df.index + 1

# --- tickets table ---
tickets = pd.DataFrame({
    "ticket_id": df["ticket_id"],
    "department": df["Department"],
    "priority": df["Priority"],
    "body_length": df["Body"].astype(str).str.len(),
})

# --- ticket_tags table: one row per ticket per tag ---
rows = []
for tid, raw in zip(df["ticket_id"], df["Tags"]):
    try:
        tags = ast.literal_eval(str(raw))
    except (ValueError, SyntaxError):
        continue
    if not isinstance(tags, list):
        continue
    for tag in tags:
        rows.append({"ticket_id": tid, "tag": str(tag).strip()})

ticket_tags = pd.DataFrame(rows)

conn = sqlite3.connect("tickets.db")
tickets.to_sql("tickets", conn, if_exists="replace", index=False)
ticket_tags.to_sql("ticket_tags", conn, if_exists="replace", index=False)

conn.execute("CREATE INDEX IF NOT EXISTS idx_tags_ticket ON ticket_tags(ticket_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_tags_tag ON ticket_tags(tag)")
conn.commit()

print(f"tickets:     {len(tickets):,} rows")
print(f"ticket_tags: {len(ticket_tags):,} rows")
print(f"unique tags: {ticket_tags['tag'].nunique()}")
conn.close()

tickets:     29,651 rows
ticket_tags: 146,390 rows
unique tags: 1650
